# 02 — Leakage-aware split design

This notebook freezes three roles:

- `train`: older headlines from non-holdout publications;
- `validation`: newest 20% within each non-holdout publication;
- `test_ood`: all records from one unseen publication per class.

The OOD set is not used for model selection. Exact normalized headlines cannot cross active
splits; a development record duplicated in the frozen OOD set is marked `excluded`.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

DATA_PATH = ROOT / "data/processed/publication_proxy_headlines.csv"

In [2]:
import hashlib
import json
import pandas as pd

from bias_dataset.modeling import (
    LABEL_ORDER, assert_split_integrity, load_modeling_data, make_pilot_split
)

SPLIT_DIR = ROOT / "data/splits"
SPLIT_PATH = SPLIT_DIR / "pilot_v1.csv"

## Load data and create the deterministic manifest

In [3]:
data = load_modeling_data(DATA_PATH)
manifest = make_pilot_split(data, validation_fraction=0.20)
manifest.head()

,record_id,source_id,weak_label,headline_hash,model_timestamp,split,split_reason
0,006c781fbb88dbc6fa7ed5496b3b20aea788c673520923...,the_hill,Center,a37fd4f3c9f72a48be6598e65c1f4a17374a2d507f3074...,2026-08-26 23:55:33+00:00,validation,newest_within_outlet
1,00a12f7b1f87d25fd4c053a077003edde839b1e2784083...,vox,Left,3c54794c14922a3ca5b49b8c44c1b5fde98aea8e559a34...,2026-08-21 20:15:18+00:00,train,older_non_holdout_record
2,00aa44240af86fd7819737d37e982b1be793a5920c6321...,fox_news,Right,f2ad76eb6ece0d7944e0d681b2f5c951430b6b108421f2...,2026-08-21 18:13:45+00:00,train,older_non_holdout_record
3,00c3c0ce76a022ef572968027a27cd7f24afb53ff8664a...,cnn,Lean Left,fb7546a5dd4d2993d233c020616cbdaacc026b6bc0002e...,2023-04-18 22:03:13+00:00,test_ood,frozen_holdout_outlet
4,00c681a1ee3f6af841285dc8af13d8bbc73c53000ba0eb...,washington_examiner,Lean Right,d8eb32707fe128679261463580c186a8a9b4fdb1bad86e...,2026-08-24 21:52:31+00:00,test_ood,frozen_holdout_outlet


## Count each split before training

In [4]:
split_counts = manifest["split"].value_counts().rename_axis("split").to_frame("records")
split_counts

,records
split,
test_ood,496
train,419
validation,118


In [5]:
split_by_class = pd.crosstab(manifest["weak_label"], manifest["split"]).reindex(LABEL_ORDER)
split_by_class

split,test_ood,train,validation
weak_label,,,
Left,153,58,15
Lean Left,44,94,26
Center,23,98,32
Lean Right,232,91,24
Right,44,78,21


Macro-F1 later gives every class equal importance even though the frozen OOD publications
contribute different record counts. Training also uses balanced class weights.

## Confirm publication isolation

In [6]:
source_allocation = (
    manifest.groupby(["weak_label", "source_id", "split"], observed=True)
    .size().rename("records").reset_index()
)
source_allocation.sort_values(["weak_label", "source_id", "split"])

,weak_label,source_id,split,records
0,Center,cs_monitor,test_ood,23
1,Center,newsnation,train,45
2,Center,newsnation,validation,12
3,Center,the_hill,train,53
4,Center,the_hill,validation,20
5,Lean Left,abc_news,train,29
6,Lean Left,abc_news,validation,8
7,Lean Left,cnn,test_ood,44
8,Lean Left,mediaite,train,44
9,Lean Left,mediaite,validation,12


In [7]:
development_sources = set(manifest.loc[manifest["split"].isin(["train", "validation"]), "source_id"])
ood_sources = set(manifest.loc[manifest["split"].eq("test_ood"), "source_id"])
print("Development outlets:", sorted(development_sources))
print("OOD outlets:", sorted(ood_sources))
print("Overlap:", development_sources & ood_sources)

Development outlets: ['abc_news', 'breitbart', 'daily_wire', 'fox_news', 'mediaite', 'nbc_news', 'newsnation', 'reason', 'the_atlantic', 'the_dispatch', 'the_hill', 'vox']
OOD outlets: ['cnn', 'cs_monitor', 'mother_jones', 'newsmax', 'washington_examiner']
Overlap: set()


## Confirm exact-headline isolation

In [8]:
active = manifest[manifest["split"].isin(["train", "validation", "test_ood"])]
hashes_crossing_splits = active.groupby("headline_hash")["split"].nunique().gt(1).sum()
print("Exact headline hashes crossing active splits:", hashes_crossing_splits)

Exact headline hashes crossing active splits: 0


In [9]:
excluded = manifest[manifest["split"].eq("excluded")]
print(f"Excluded cross-boundary duplicate records: {len(excluded)}")
excluded[["record_id", "source_id", "weak_label", "split_reason"]]

Excluded cross-boundary duplicate records: 0


,record_id,source_id,weak_label,split_reason


## Run executable leakage assertions

In [10]:
assert_split_integrity(manifest)
print("PASS: unique IDs, all five labels, outlet isolation, and exact-hash isolation.")

PASS: unique IDs, all five labels, outlet isolation, and exact-hash isolation.


## Inspect the within-outlet time boundary

In [11]:
boundaries = (
    manifest[manifest["split"].isin(["train", "validation"])]
    .groupby(["source_id", "split"], observed=True)["model_timestamp"]
    .agg(["min", "max", "count"])
)
boundaries

min                       max  \
source_id    split                                                            
abc_news     train      2026-08-20 02:26:16+00:00 2026-08-26 12:34:14+00:00   
             validation 2026-08-26 15:13:28+00:00 2026-08-27 09:06:20+00:00   
breitbart    train      2026-08-26 07:42:47+00:00 2026-08-26 23:12:34+00:00   
             validation 2026-08-26 23:14:27+00:00 2026-08-27 03:13:16+00:00   
daily_wire   train      2026-08-26 04:00:29+00:00 2026-08-26 15:39:46+00:00   
             validation 2026-08-26 16:00:21+00:00 2026-08-26 17:47:26+00:00   
fox_news     train      2026-08-19 11:15:00+00:00 2026-08-26 22:42:57+00:00   
             validation 2026-08-26 23:19:37+00:00 2026-08-27 09:00:52+00:00   
mediaite     train      2026-08-21 02:29:01+00:00 2026-08-26 19:48:36+00:00   
             validation 2026-08-26 19:48:53+00:00 2026-08-27 01:15:51+00:00   
nbc_news     train      2026-07-20 21:52:47+00:00 2026-08-26 17:17:23+00:00   
             validation 2026-08-26 20:24:57+00:00 2026-08-27 01:19:18+00:00   
newsnation   train      2026-08-21 11:55:42+00:00 2026-08-26 14:59:42+00:00   
             validation 2026-08-26 16:32:39+00:00 2026-08-27 03:43:18+00:00   
reason       train      2026-08-11 11:00:36+00:00 2026-08-23 11:00:29+00:00   
             validation 2026-08-24 11:00:31+00:00 2026-08-26 22:02:56+00:00   
the_atlantic train      2026-07-07 15:30:00+00:00 2026-08-25 11:00:00+00:00   
             validation 2026-08-25 11:00:00+00:00 2026-08-26 22:18:48+00:00   
the_dispatch train      2026-08-07 05:02:00+00:00 2026-08-22 12:00:00+00:00   
             validation 2026-08-25 05:00:00+00:00 2026-08-27 07:00:00+00:00   
the_hill     train      2026-08-25 12:03:00+00:00 2026-08-26 20:10:37+00:00   
             validation 2026-08-26 14:33:04+00:00 2026-08-27 03:16:41+00:00   
vox          train      2026-07-31 21:54:47+00:00 2026-08-24 21:34:23+00:00   
             validation 2026-08-24 23:15:45+00:00 2026-08-25 22:55:09+00:00   

                         count  
source_id    split              
abc_news     train          29  
             validation      8  
breitbart    train          22  
             validation      6  
daily_wire   train          19  
             validation      5  
fox_news     train          37  
             validation     10  
mediaite     train          44  
             validation     12  
nbc_news     train          21  
             validation      6  
newsnation   train          45  
             validation     12  
reason       train          82  
             validation     21  
the_atlantic train          47  
             validation     12  
the_dispatch train           9  
             validation      3  
the_hill     train          53  
             validation     20  
vox          train          11  
             validation      3

An exact duplicate can promote an older training record into validation. This is intentional:
duplicate isolation takes precedence over a perfectly sharp date boundary.

## Save the versioned manifest and checksum

In [12]:
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
manifest.to_csv(SPLIT_PATH, index=False)

dataset_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
print("Manifest:", SPLIT_PATH.relative_to(ROOT))
print("Dataset SHA-256:", dataset_sha256)

Manifest: data/splits/pilot_v1.csv
Dataset SHA-256: 5551ced956bd38a8549a2b39a12739e91d841cf2f333a20b462390805db93a5b


In [13]:
metadata = {
    "split_version": "pilot_v1",
    "dataset_sha256": dataset_sha256,
    "validation_fraction": 0.20,
    "counts": manifest["split"].value_counts().sort_index().to_dict(),
    "ood_sources": sorted(ood_sources),
}
metadata_path = SPLIT_DIR / "pilot_v1_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2) + "\n")
metadata

{'split_version': 'pilot_v1',
 'dataset_sha256': '5551ced956bd38a8549a2b39a12739e91d841cf2f333a20b462390805db93a5b',
 'validation_fraction': 0.2,
 'counts': {'test_ood': 496, 'train': 419, 'validation': 118},
 'ood_sources': ['cnn',
  'cs_monitor',
  'mother_jones',
  'newsmax',
  'washington_examiner']}